# Feature Engineered Model

This notebook combines the most important features from Model A (BMI, Age) with Model B clinical factors through strategic feature engineering.

**Model A Top Features (based on feature importance):**
- BMI (1367) - Body Mass Index
- Age (1157) - Patient age

**Model B Clinical Features:**
- Age, BMI, Gender
- Cholesterol Level
- Glucose Level
- Systolic Blood Pressure
- Diastolic Blood Pressure
- Is Minority (for sample weighting)

**Feature Engineering Strategy:**
- Uses Model B as the base (clinical health factors)
- Creates interaction features between Age/BMI and clinical factors
- Focuses on clinically validated cardiovascular risk pathways

6. **Age × Glucose** - Age-related glucose metabolism decline

**Engineered Features (Age & BMI based):**5. **BMI × Glucose** - Metabolic syndrome indicator

1. **Age × BMI** - Joint obesity-aging cardiovascular risk4. **Age × Cholesterol** - Cumulative atherogenic burden

2. **Age × Systolic BP** - Age-related arterial stiffness and hypertension3. **BMI × Systolic BP** - Obesity-induced hypertension pathway

## Load Datasets

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, roc_curve, precision_recall_curve, auc, classification_report
)

In [ ]:
# Load Model A dataset (lifestyle factors)
dataset_a = pd.read_csv("Model A Dataset/lifestyle_dataset.csv")
print("Model A Dataset Shape:", dataset_a.shape)
print("Model A Columns:", dataset_a.columns.tolist())
print("\nModel A Preview:")
dataset_a.head()

In [ ]:
# Load Model B dataset (health factors)
dataset_b = pd.read_csv("Model B Dataset/healthFactors_dataset_with_indicator.csv")
print("Model B Dataset Shape:", dataset_b.shape)
print("Model B Columns:", dataset_b.columns.tolist())
print("\nModel B Preview:")
dataset_b.head()

## Feature Engineering Analysis

Check overlap between datasets and identify unique features from Model A

In [ ]:
# Identify common features (excluding target)
model_a_features = set(dataset_a.columns) - {'Cardiovascular Disease'}
model_b_features = set(dataset_b.columns) - {'Cardiovascular Disease', 'Is Minority'}

common_features = model_a_features & model_b_features
unique_a_features = model_a_features - model_b_features
unique_b_features = model_b_features - model_a_features

print("Common Features (in both datasets):")
print(sorted(common_features))
print(f"\nUnique to Model A (lifestyle factors):")
print(sorted(unique_a_features))
print(f"\nUnique to Model B (clinical factors):")
print(sorted(unique_b_features))

## Strategy 1: Model B + Top Lifestyle Features

Add the most important unique lifestyle features from Model A (Smoking Status, Physical Activity, Alcohol Intake) to Model B

In [ ]:
# Since datasets have different samples, we'll use Model B as base and add lifestyle features
# This requires identifying matching patients or using Model B independently with added synthetic features

# For now, let's use Model B with its existing features + analyze if we can merge
print("Model A rows:", len(dataset_a))
print("Model B rows:", len(dataset_b))
print("\nModel A target distribution:")
print(dataset_a['Cardiovascular Disease'].value_counts())
print("\nModel B target distribution:")
print(dataset_b['Cardiovascular Disease'].value_counts())

## Feature Engineering: Age & BMI Interactions

Create clinically-validated interaction features between the top Model A features (Age, BMI) and Model B clinical factors.

All engineered features leverage Age and BMI's predictive power with clinical health metrics.

In [ ]:
# Create feature-engineered dataset from Model B
# Add interaction features between top Model A features (Age, BMI) and Model B clinical factors

dataset_enhanced = dataset_b.copy()

# ============ TIER 1: Highest Clinical Significance ============
# 1. Age × BMI - Joint effect of aging and obesity on CVD risk
dataset_enhanced['Age_BMI_Interaction'] = dataset_enhanced['Age'] * dataset_enhanced['BMI']

# 2. Age × Systolic BP - Age-related arterial stiffness and hypertension
dataset_enhanced['Age_SystolicBP'] = dataset_enhanced['Age'] * dataset_enhanced['Systolic Blood Pressure']

# 3. BMI × Systolic BP - Obesity-induced hypertension pathway
dataset_enhanced['BMI_SystolicBP'] = dataset_enhanced['BMI'] * dataset_enhanced['Systolic Blood Pressure']

# 4. Age × Cholesterol - Cumulative atherogenic burden over time
dataset_enhanced['Age_Cholesterol'] = dataset_enhanced['Age'] * dataset_enhanced['Cholesterol Level']

# ============ TIER 2: Important Metabolic Interactions ============
# 5. BMI × Glucose - Metabolic syndrome and insulin resistance indicator
dataset_enhanced['BMI_Glucose'] = dataset_enhanced['BMI'] * dataset_enhanced['Glucose Level']

# 6. Age × Glucose - Age-related glucose metabolism decline
dataset_enhanced['Age_Glucose'] = dataset_enhanced['Age'] * dataset_enhanced['Glucose Level']

print("="*60)
print("Enhanced Dataset with Age/BMI-based Feature Engineering")
print("="*60)
print(f"Original Dataset Shape: {dataset_b.shape}")
print(f"Enhanced Dataset Shape: {dataset_enhanced.shape}")
print(f"\nNew Features Added: {dataset_enhanced.shape[1] - dataset_b.shape[1]}")

dataset_enhanced.head()

new_features = sorted(set(dataset_enhanced.columns) - set(dataset_b.columns))print("\nEnhanced Dataset Preview:")

print("\nEngineered Features:")

for i, feat in enumerate(new_features, 1):    print(f"  {i}. {feat}")

## Prepare Data for Training

In [ ]:
# Prepare features and target
X = dataset_enhanced.drop(['Cardiovascular Disease', 'Is Minority'], axis=1)
Y = dataset_enhanced['Cardiovascular Disease']

# Create sample weights (10x weight for minority class)
sample_weights = dataset_enhanced['Is Minority'].apply(lambda x: 10 if x == 1 else 1)

print("Feature Matrix Shape:", X.shape)
print("Features:", X.columns.tolist())
print("\nTarget Distribution:")
print(Y.value_counts())
print("\nSample Weights Distribution:")
print(sample_weights.value_counts())

In [ ]:
# Train-test split with stratification
X_train, X_test, Y_train, Y_test, weights_train, weights_test = train_test_split(
    X, Y, sample_weights,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## Train Enhanced LightGBM Model

In [ ]:
# Train LightGBM with enhanced features
lgbm_enhanced = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)

lgbm_enhanced.fit(
    X_train,
    Y_train.to_numpy(),
    sample_weight=weights_train.to_numpy()
)

# Predictions
y_pred = lgbm_enhanced.predict(X_test)
y_prob = lgbm_enhanced.predict_proba(X_test)[:, 1]

print("✅ Enhanced LightGBM model trained successfully!")

## Model Evaluation

In [ ]:
# Calculate metrics
acc = accuracy_score(Y_test, y_pred)
prec = precision_score(Y_test, y_pred)
rec = recall_score(Y_test, y_pred)
f1 = f1_score(Y_test, y_pred)
roc_auc = roc_auc_score(Y_test, y_prob)

precision_curve, recall_curve, _ = precision_recall_curve(Y_test, y_prob)
pr_auc = auc(recall_curve, precision_curve)

print("="*50)
print("Enhanced Model Performance Metrics")
print("="*50)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("\nClassification Report:")
print(classification_report(Y_test, y_pred))

In [ ]:
# Visualize ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
fpr, tpr, _ = roc_curve(Y_test, y_prob)
axes[0].plot(fpr, tpr, label=f'Enhanced Model (AUC={roc_auc:.4f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curve - Enhanced Model', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall_curve, precision_curve, label=f'Enhanced Model (AUC={pr_auc:.4f})', linewidth=2)
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve - Enhanced Model', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## K-Fold Cross Validation

In [ ]:
# K-Fold Cross Validation
X_full = dataset_enhanced.drop(['Cardiovascular Disease', 'Is Minority'], axis=1)
Y_full = dataset_enhanced['Cardiovascular Disease']
weights_full = dataset_enhanced['Is Minority'].apply(lambda x: 10 if x == 1 else 1)

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

fold_metrics = {
    'Fold': [],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1-Score': [],
    'ROC-AUC': [],
    'PR-AUC': []
}

print(f"Performing {n_splits}-Fold Cross Validation on Enhanced Model...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_full), 1):
    X_train_fold = X_full.iloc[train_idx]
    X_val_fold = X_full.iloc[val_idx]
    Y_train_fold = Y_full.iloc[train_idx]
    Y_val_fold = Y_full.iloc[val_idx]
    weights_train_fold = weights_full.iloc[train_idx]
    
    lgbm_kfold = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=100,
        learning_rate=0.05,
        random_state=42,
        verbose=-1
    )
    lgbm_kfold.fit(
        X_train_fold,
        Y_train_fold.to_numpy(),
        sample_weight=weights_train_fold.to_numpy()
    )
    
    y_pred_fold = lgbm_kfold.predict(X_val_fold)
    y_prob_fold = lgbm_kfold.predict_proba(X_val_fold)[:, 1]
    
    acc = accuracy_score(Y_val_fold, y_pred_fold)
    prec = precision_score(Y_val_fold, y_pred_fold)
    rec = recall_score(Y_val_fold, y_pred_fold)
    f1 = f1_score(Y_val_fold, y_pred_fold)
    roc_auc = roc_auc_score(Y_val_fold, y_prob_fold)
    
    precision_curve, recall_curve, _ = precision_recall_curve(Y_val_fold, y_prob_fold)
    pr_auc = auc(recall_curve, precision_curve)
    
    fold_metrics['Fold'].append(fold)
    fold_metrics['Accuracy'].append(acc)
    fold_metrics['Precision'].append(prec)
    fold_metrics['Recall'].append(rec)
    fold_metrics['F1-Score'].append(f1)
    fold_metrics['ROC-AUC'].append(roc_auc)
    fold_metrics['PR-AUC'].append(pr_auc)
    
    print(f"Fold {fold}:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")
    print(f"  PR-AUC:    {pr_auc:.4f}\n")

kfold_results_df = pd.DataFrame(fold_metrics)
print(kfold_results_df.to_string(index=False))

In [ ]:
# Average metrics across folds
print("="*50)
print("Average Metrics Across All Folds:")
print("="*50)
print(f"Accuracy:  {np.mean(fold_metrics['Accuracy']):.4f} ± {np.std(fold_metrics['Accuracy']):.4f}")
print(f"Precision: {np.mean(fold_metrics['Precision']):.4f} ± {np.std(fold_metrics['Precision']):.4f}")
print(f"Recall:    {np.mean(fold_metrics['Recall']):.4f} ± {np.std(fold_metrics['Recall']):.4f}")
print(f"F1-Score:  {np.mean(fold_metrics['F1-Score']):.4f} ± {np.std(fold_metrics['F1-Score']):.4f}")
print(f"ROC-AUC:   {np.mean(fold_metrics['ROC-AUC']):.4f} ± {np.std(fold_metrics['ROC-AUC']):.4f}")
print(f"PR-AUC:    {np.mean(fold_metrics['PR-AUC']):.4f} ± {np.std(fold_metrics['PR-AUC']):.4f}")

# Visualize fold performance
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC']
x = np.arange(n_splits)
width = 0.12

for i, metric in enumerate(metrics_to_plot):
    axes[0].bar(x + i * width, fold_metrics[metric], width, label=metric)

axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Score')
axes[0].set_title('Enhanced Model Performance per Fold')
axes[0].set_xticks(x + width * 2.5)
axes[0].set_xticklabels([f'Fold {i+1}' for i in range(n_splits)])
axes[0].legend(loc='lower right')
axes[0].set_ylim(0, 1.1)

axes[1].boxplot([fold_metrics[m] for m in metrics_to_plot], labels=metrics_to_plot)
axes[1].set_ylabel('Score')
axes[1].set_title('Distribution of Metrics Across Folds')
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

## Feature Importance Analysis

In [ ]:
# Feature importance for enhanced model
feature_importance = pd.DataFrame({
    'Feature': X_full.columns,
    'Importance': lgbm_enhanced.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance - Enhanced Model:")
print(feature_importance.to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(12, 8))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance - Enhanced Model', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Save Enhanced Model

In [ ]:
import os
import joblib

os.makedirs("saved_models_tausif", exist_ok=True)

# Train final model on full dataset
lgbm_final = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)
lgbm_final.fit(
    X_full,
    Y_full.to_numpy(),
    sample_weight=weights_full.to_numpy()
)

# Save models
joblib.dump(lgbm_final, "saved_models_tausif/lightgbm_enhanced_model.joblib")
lgbm_final.booster_.save_model("saved_models_tausif/lightgbm_enhanced_model.txt")

# Save metrics
kfold_results_df.to_csv("saved_models_tausif/enhanced_model_kfold_metrics.csv", index=False)
feature_importance.to_csv("saved_models_tausif/enhanced_model_feature_importance.csv", index=False)

# Save single test metrics
test_metrics = pd.DataFrame([{
    'Model': 'Enhanced LightGBM',
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1-Score': f1,
    'ROC-AUC': roc_auc,
    'PR-AUC': pr_auc
}])
test_metrics.to_csv("saved_models_tausif/enhanced_model_test_metrics.csv", index=False)

print("✅ Enhanced model saved successfully!")
print("   - saved_models_tausif/lightgbm_enhanced_model.joblib")
print("   - saved_models_tausif/lightgbm_enhanced_model.txt")
print("   - saved_models_tausif/enhanced_model_kfold_metrics.csv")
print("   - saved_models_tausif/enhanced_model_feature_importance.csv")
print("   - saved_models_tausif/enhanced_model_test_metrics.csv")